# สร้างและตรวจ CENTRAL-5M

ใช้ไฟล์ Parquet จาก final_scale เป็น train และนำ validation/test จาก CENTRAL มาจัดแพ็ก รันจากบนลงล่างหลังเตรียมข้อมูลต้นทางทั้งสองชุด


## 1. เตรียม path และเครื่องมือตรวจ checksum

กำหนดตำแหน่งของ final_scale, CENTRAL และ CENTRAL-5M รวมถึงชื่อคลาสที่ใช้แสดงผล


In [ ]:
from pathlib import Path

PIPELINE_DIR = next(
    candidate
    for ancestor in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    for candidate in (ancestor, ancestor / "data-pipeline")
    if (candidate / "CENTRAL" / "feature_contract.json").is_file()
)

import hashlib
import json
import os
import shutil
import time

import numpy as np
import pandas as pd

# Paths
PROJECT_ROOT = PIPELINE_DIR.parent
DEFAULT_SOURCE_DIR = PROJECT_ROOT / "experiments/scaling_and_domain_shift/data/final_scale"
DEFAULT_CENTRAL_DIR = PIPELINE_DIR / "CENTRAL"
DEFAULT_TARGET_DIR = PIPELINE_DIR / "CENTRAL-5M"

CLASS_NAMES = [
    "Benign / Normal",
    "Denial of Service (DoS)",
    "Reconnaissance & Probing",
    "Exploits & PrivEsc",
    "Credential Abuse (R2L)",
    "Malware, C2 & Botnet",
]

TACTICS = [
    "Baseline Operations",
    "TA0040: Impact",
    "TA0043/TA0007: Recon & Discovery",
    "TA0001/TA0004: Initial Access & PrivEsc",
    "TA0006: Credential Access",
    "TA0011/TA0008: C2 & Lateral Movement",
]


def compute_sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(1024 * 1024):
            h.update(chunk)
    return h.hexdigest()




print(f"Pipeline root located at: {PIPELINE_DIR}")


## 2. เปิดดู schema และทะเบียนชุดข้อมูล

อ่าน feature_contract.json และ dataset_registry.json เพื่อดูฟีเจอร์และแหล่งข้อมูลที่คาดว่าจะใช้


In [2]:
central_5m_dir = PIPELINE_DIR / 'CENTRAL-5M'
with open(central_5m_dir / 'feature_contract.json') as f:
    contract = json.load(f)
with open(central_5m_dir / 'dataset_registry.json') as f:
    registry = json.load(f)

print(f"Pipeline Name: {contract['pipeline_name']} (Contract v{contract['contract_version']})")
print(f"Supported Datasets: {contract['supported_datasets']}")
print(f"Expected Features ({contract['joint_model_profile']['expected_model_feature_count']}):")
for k, v in contract['joint_model_profile']['numeric_features'].items():
    print(f"  - {k:25s}: {v}")
for k, v in contract['joint_model_profile']['categorical_inputs'].items():
    print(f"  - {k:25s}: {v}")


Pipeline Name: CENTRAL-5M (Contract v6)
Supported Datasets: ['NSL-KDD', 'UNSW-NB15', 'CIC-IDS2017', 'BoT-IoT', 'CIC-IDS2018', 'ToN-IoT']
Expected Features (19):
  - log_duration             : log1p(duration)
  - log_bytes_forward        : log1p(bytes_forward)
  - log_bytes_backward       : log1p(bytes_backward)
  - signed_log_bytes_ratio   : log1p(bytes_forward) - log1p(bytes_backward)
  - protocol                 : ['tcp', 'udp', 'icmp', 'other']
  - service_group            : ['dns', 'ftp', 'http', 'https_ssl', 'smtp', 'ssh', 'other']
  - conn_state_group         : ['established', 'no-reply', 'reset', 'other']


## 3. เตรียมโฟลเดอร์ผลลัพธ์และตรวจไฟล์ต้นทาง

กำหนดโฟลเดอร์สำหรับแต่ละ split และตรวจว่า final_scale มี features, labels และ metadata ครบ


In [ ]:
source_dir = DEFAULT_SOURCE_DIR
central_dir = DEFAULT_CENTRAL_DIR
target_dir = central_5m_dir

start_time = time.time()
source_dir = Path(source_dir)
central_dir = Path(central_dir)
target_dir = Path(target_dir)

out_dir = target_dir / "output"
harmonized_dir = out_dir / "harmonized"
train_dir = out_dir / "splits" / "train"
val_dir = out_dir / "splits" / "validation"
test_dir = out_dir / "splits" / "test"
feat_eng_dir = out_dir / "feature_engineering"
reports_dir = out_dir / "reports"

print("=" * 70)
print("Building data-pipeline/CENTRAL-5M from final_scale & canonical CENTRAL")
print(f"Source Directory : {source_dir}")
print(f"Target Directory : {target_dir}")
print("=" * 70)

# 1. Setup Directories
for d in [harmonized_dir, train_dir, val_dir, test_dir, feat_eng_dir, reports_dir]:
    d.mkdir(parents=True, exist_ok=True)

# 2. Source Files
src_feat = source_dir / "central_final_features_19.parquet"
src_lbl = source_dir / "central_final_labels.parquet"
src_meta = source_dir / "central_final_metadata.parquet"

assert src_feat.exists(), f"Source feature parquet missing: {src_feat}"
assert src_lbl.exists(), f"Source label parquet missing: {src_lbl}"
assert src_meta.exists(), f"Source metadata parquet missing: {src_meta}"


## 4. สร้าง harmonized และ train

คัดลอกตาราง Parquet จาก final_scale เพื่อใช้เป็นข้อมูลฝึก


In [ ]:
print("[1/5] Materializing Harmonized dataset (5.04M records)...")
shutil.copy2(src_feat, harmonized_dir / "features.parquet")
shutil.copy2(src_lbl, harmonized_dir / "labels.parquet")
shutil.copy2(src_meta, harmonized_dir / "metadata.parquet")

print("[2/5] Materializing Train split (5.04M records)...")
shutil.copy2(src_feat, train_dir / "features.parquet")
shutil.copy2(src_lbl, train_dir / "labels.parquet")
shutil.copy2(src_meta, train_dir / "metadata.parquet")


## 5. สร้าง validation จาก CENTRAL

อ่านฟีเจอร์ที่ encode แล้วพร้อม labels และ metadata ของ validation จาก CENTRAL แล้วเขียนเป็น Parquet


In [ ]:
print("[3/5] Materializing Canonical Validation split (121,199 records)...")
val_feat_src = central_dir / "output/feature_engineering/validation/features.csv"
val_lbl_src = central_dir / "output/splits/validation/labels.csv"
val_meta_src = central_dir / "output/splits/validation/metadata.csv"

val_f_df = pd.read_csv(val_feat_src, low_memory=False)
val_l_df = pd.read_csv(val_lbl_src, low_memory=False)
val_m_df = pd.read_csv(val_meta_src, low_memory=False)

val_f_df.to_parquet(val_dir / "features.parquet", index=False, engine="pyarrow")
val_l_df.to_parquet(val_dir / "labels.parquet", index=False, engine="pyarrow")
val_m_df.to_parquet(val_dir / "metadata.parquet", index=False, engine="pyarrow")


## 6. สร้าง test จาก CENTRAL

อ่านฟีเจอร์ที่ encode แล้วพร้อม labels และ metadata ของ test จาก CENTRAL แล้วเขียนเป็น Parquet


In [ ]:
print("[4/5] Materializing Canonical Test split (143,095 records)...")
test_feat_src = central_dir / "output/feature_engineering/test/features.csv"
test_lbl_src = central_dir / "output/splits/test/labels.csv"
test_meta_src = central_dir / "output/splits/test/metadata.csv"

test_f_df = pd.read_csv(test_feat_src, low_memory=False)
test_l_df = pd.read_csv(test_lbl_src, low_memory=False)
test_m_df = pd.read_csv(test_meta_src, low_memory=False)

test_f_df.to_parquet(test_dir / "features.parquet", index=False, engine="pyarrow")
test_l_df.to_parquet(test_dir / "labels.parquet", index=False, engine="pyarrow")
test_m_df.to_parquet(test_dir / "metadata.parquet", index=False, engine="pyarrow")


## 7. จัดแพ็ก preprocessor และเขียน schema

คัดลอก preprocessor จาก CENTRAL แล้วบันทึกชื่อ ชนิด และจำนวนฟีเจอร์ที่ใช้จริง


In [ ]:
print("[5/5] Packaging Feature Engineering & Generating Reports...")
prep_src = central_dir / "output/feature_engineering/preprocessor.joblib"
shutil.copy2(prep_src, feat_eng_dir / "preprocessor.joblib")

# Read train metadata & labels for audit
train_l_df = pd.read_parquet(train_dir / "labels.parquet")
train_m_df = pd.read_parquet(train_dir / "metadata.parquet")
train_f_df = pd.read_parquet(train_dir / "features.parquet")

feature_cols = [c for c in train_f_df.columns if c != "record_id"]
feature_schema = {
    "schema_version": 6,
    "dataset_name": "CENTRAL-5M",
    "model_feature_names": feature_cols,
    "feature_count": len(feature_cols),
    "total_train_flows": len(train_f_df),
    "total_validation_flows": len(val_f_df),
    "total_test_flows": len(test_f_df),
    "fit_split": "train",
    "numeric_scaling": "StandardScaler fitted on train (joint_v2)",
    "categorical_encoding": "OneHotEncoder fitted on train (joint_v2)",
    "input_types": {col: str(train_f_df[col].dtype) for col in train_f_df.columns},
}
with open(feat_eng_dir / "feature_schema.json", "w") as f:
    json.dump(feature_schema, f, indent=2)


## 8. สรุปที่มาของข้อมูลและ mapping ของ label

นับจำนวนแถวตามชุดข้อมูลและ split แล้วเขียนตารางเทียบ label ต้นทางกับคลาสกลาง


In [ ]:
# Source splits breakdown
splits_records = []
for src, cnt in train_m_df["source_dataset"].value_counts().items():
    splits_records.append({"split": "train", "source_dataset": src, "rows": int(cnt)})
for src, cnt in val_m_df["source_dataset"].value_counts().items():
    splits_records.append({"split": "validation", "source_dataset": src, "rows": int(cnt)})
for src, cnt in test_m_df["source_dataset"].value_counts().items():
    splits_records.append({"split": "test", "source_dataset": src, "rows": int(cnt)})

source_splits_df = pd.DataFrame(splits_records)
source_splits_df.to_csv(reports_dir / "source_splits.csv", index=False)

# Label mapping crosstab
label_map_df = train_l_df.groupby(["source_dataset", "label_raw", "attack_category_id"]).size().unstack(fill_value=0)
label_map_df.reset_index().to_csv(reports_dir / "label_mapping.csv", index=False)


## 9. สร้าง manifest และรายงาน audit

คำนวณ SHA-256 ของผลลัพธ์และบันทึกจำนวนแถว คลาส จำนวนฟีเจอร์ และผลตรวจความถูกต้องลงใน audit.json


In [ ]:
# Checksums
files_to_audit = {
    "harmonized_features": harmonized_dir / "features.parquet",
    "harmonized_labels": harmonized_dir / "labels.parquet",
    "harmonized_metadata": harmonized_dir / "metadata.parquet",
    "train_features": train_dir / "features.parquet",
    "train_labels": train_dir / "labels.parquet",
    "train_metadata": train_dir / "metadata.parquet",
    "validation_features": val_dir / "features.parquet",
    "validation_labels": val_dir / "labels.parquet",
    "validation_metadata": val_dir / "metadata.parquet",
    "test_features": test_dir / "features.parquet",
    "test_labels": test_dir / "labels.parquet",
    "test_metadata": test_dir / "metadata.parquet",
    "preprocessor": feat_eng_dir / "preprocessor.joblib",
}

file_manifest = {}
for name, path in files_to_audit.items():
    file_manifest[name] = {
        "filename": path.name,
        "path": str(path.relative_to(target_dir)),
        "size_bytes": path.stat().st_size,
        "size_mb": round(path.stat().st_size / (1024 * 1024), 2),
        "sha256": compute_sha256(path),
    }

train_class_counts = {int(k): int(v) for k, v in train_l_df["attack_category_id"].value_counts().sort_index().items()}
val_class_counts = {int(k): int(v) for k, v in val_l_df["attack_category_id"].value_counts().sort_index().items()}
test_class_counts = {int(k): int(v) for k, v in test_l_df["attack_category_id"].value_counts().sort_index().items()}

train_binary_counts = {int(k): int(v) for k, v in train_l_df["label_binary"].value_counts().sort_index().items()}
val_binary_counts = {int(k): int(v) for k, v in val_l_df["label_binary"].value_counts().sort_index().items()}
test_binary_counts = {int(k): int(v) for k, v in test_l_df["label_binary"].value_counts().sort_index().items()}

audit_data = {
    "pipeline_name": "CENTRAL-5M",
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "status": "passed",
    "format": "Parquet-First",
    "total_records": len(train_f_df) + len(val_f_df) + len(test_f_df),
    "source_datasets": {
        "CIC-IDS2017": int((train_m_df["source_dataset"] == "CIC-IDS2017").sum()),
        "BoT-IoT": int((train_m_df["source_dataset"] == "BoT-IoT").sum()),
        "CIC-IDS2018": int((train_m_df["source_dataset"] == "CIC-IDS2018").sum()),
        "UNSW-NB15": int((train_m_df["source_dataset"] == "UNSW-NB15").sum()),
        "ToN-IoT": int((train_m_df["source_dataset"] == "ToN-IoT").sum()),
        "NSL-KDD": int((train_m_df["source_dataset"] == "NSL-KDD").sum()),
    },
    "splits": {
        "train": {
            "rows": len(train_f_df),
            "features": len(feature_cols),
            "classes": train_class_counts,
            "binary": train_binary_counts,
        },
        "validation": {
            "rows": len(val_f_df),
            "features": len(feature_cols),
            "classes": val_class_counts,
            "binary": val_binary_counts,
        },
        "test": {
            "rows": len(test_f_df),
            "features": len(feature_cols),
            "classes": test_class_counts,
            "binary": test_binary_counts,
        },
    },
    "file_manifest": file_manifest,
    "invariants": {
        "train_row_count_ge_5m": len(train_f_df) >= 5_000_000,
        "validation_row_count_canonical": len(val_f_df) == 121_199,
        "test_row_count_canonical": len(test_f_df) == 143_095,
        "feature_count_exact_19": len(feature_cols) == 19,
        "all_features_finite": bool(np.all(np.isfinite(train_f_df[feature_cols].to_numpy()[:10000]))),
        "no_duplicate_train_record_ids": bool(train_f_df["record_id"].nunique() == len(train_f_df)),
    },
}

with open(reports_dir / "audit.json", "w") as f:
    json.dump(audit_data, f, indent=2)

elapsed = round(time.time() - start_time, 2)
print("=" * 70)
print(f"SUCCESS: CENTRAL-5M pipeline successfully materialized in {elapsed}s!")
print(f"Train rows     : {len(train_f_df):,}")
print(f"Validation rows: {len(val_f_df):,}")
print(f"Test rows      : {len(test_f_df):,}")
print("=" * 70)
audit_result = audit_data
print(f"Build Status: {audit_result['status']}")
print(f"Total Multi-domain Records: {audit_result['total_records']:,}")


## 10. ดูจำนวนแถวตาม split และแหล่งข้อมูล

เปิด source_splits.csv เพื่อตรวจสัดส่วนข้อมูลจากแต่ละชุด


In [4]:
splits_df = pd.read_csv(central_5m_dir / 'output/reports/source_splits.csv')
print(splits_df.to_string(index=False))


     split source_dataset    rows
     train    CIC-IDS2017 2000000
     train        BoT-IoT 1800000
     train    CIC-IDS2018  619346
     train      UNSW-NB15  257673
     train        ToN-IoT  211043
     train        NSL-KDD  148517
validation      UNSW-NB15   49439
validation        BoT-IoT   29912
validation    CIC-IDS2017   28590
validation        NSL-KDD   13258
      test      UNSW-NB15   82332
      test        NSL-KDD   22544
      test        BoT-IoT   20000
      test    CIC-IDS2017   18219


## 11. ดูการกระจายคลาสของ train

นับจำนวนแถวของแต่ละคลาสจาก labels.parquet


In [5]:
train_lbl = pd.read_parquet(central_5m_dir / 'output/splits/train/labels.parquet')
class_counts = train_lbl['attack_category_id'].value_counts().sort_index()

print(f"{'ID':<4} | {'Category Name':<30} | {'Tactic':<35} | {'Flow Count':<12} | {'Percentage'}")
print("-" * 95)
total_train = len(train_lbl)
for cid, count in class_counts.items():
    name = CLASS_NAMES[cid] if 0 <= cid < len(CLASS_NAMES) else 'Unknown'
    tactic = TACTICS[cid] if 0 <= cid < len(TACTICS) else 'Unknown'
    pct = (count / total_train) * 100
    print(f"{cid:<4} | {name:<30} | {tactic:<35} | {count:<12,} | {pct:.2f}%")

norm_count = (train_lbl['label_binary'] == 0).sum()
att_count = (train_lbl['label_binary'] == 1).sum()
print("\nBinary Classification:")
print(f"  - Benign / Normal: {norm_count:,} ({norm_count/total_train*100:.2f}%)")
print(f"  - Attack Flows   : {att_count:,} ({att_count/total_train*100:.2f}%)")


ID   | Category Name                  | Tactic                              | Flow Count   | Percentage
-----------------------------------------------------------------------------------------------
0    | Benign / Normal                | Baseline Operations                 | 2,300,531    | 45.68%
1    | Denial of Service (DoS)        | TA0040: Impact                      | 2,122,048    | 42.13%
2    | Reconnaissance & Probing       | TA0043/TA0007: Recon & Discovery    | 268,792      | 5.34%
3    | Exploits & PrivEsc             | TA0001/TA0004: Initial Access & PrivEsc | 106,872      | 2.12%
4    | Credential Abuse (R2L)         | TA0006: Credential Access           | 130,093      | 2.58%
5    | Malware, C2 & Botnet           | TA0011/TA0008: C2 & Lateral Movement | 108,243      | 2.15%

Binary Classification:
  - Benign / Normal: 2,300,531 (45.68%)
  - Attack Flows   : 2,736,048 (54.32%)


## 12. ดูตัวอย่างฟีเจอร์ที่สร้างแล้ว

เปิดตารางฟีเจอร์ของ train และดูตัวอย่างแถวแรก


In [6]:
train_feat = pd.read_parquet(central_5m_dir / 'output/splits/train/features.parquet')
print(f"Features shape: {train_feat.shape}")
train_feat.head(5)


Features shape: (5036579, 20)


,record_id,log_duration,log_bytes_forward,log_bytes_backward,signed_log_bytes_ratio,proto_tcp,proto_udp,proto_icmp,proto_other,service_dns,service_ftp,service_http,service_https_ssl,service_smtp,service_ssh,service_other,state_established,state_no-reply,state_reset,state_other
0,NSL-KDD:KDDTrain:000000000,-0.69799,0.477092,-0.882694,1.278303,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,NSL-KDD:KDDTrain:000000001,-0.69799,-0.008192,-0.882694,0.921804,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
2,NSL-KDD:KDDTrain:000000002,-0.69799,-2.012901,-0.882694,-0.550891,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
3,NSL-KDD:KDDTrain:000000003,-0.69799,0.176838,1.645815,-1.600051,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,NSL-KDD:KDDTrain:000000004,-0.69799,0.115488,0.813776,-0.770541,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


## 13. ตรวจผล audit

เปิด audit.json และแสดงสถานะของเงื่อนไขที่ตรวจไว้


In [7]:
with open(central_5m_dir / 'output/reports/audit.json') as f:
    audit = json.load(f)

print("Audit Invariants Verification:")
all_passed = True
for inv, passed in audit['invariants'].items():
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_passed = False
    print(f"  [{status}] {inv}")

assert all_passed, "One or more invariants failed!"
print("\nAll CENTRAL-5M pipeline invariants successfully verified.")


Audit Invariants Verification:
  [PASS] train_row_count_ge_5m
  [PASS] validation_row_count_canonical
  [PASS] test_row_count_canonical
  [PASS] feature_count_exact_19
  [PASS] all_features_finite
  [PASS] no_duplicate_train_record_ids

All CENTRAL-5M pipeline invariants successfully verified.
